[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivastar/ml_for_astro_notebooks/blob/main/session3b_student.ipynb)

# Session 3b — Quiescent or Star-Forming? Classifying Galaxies in 3D-HST

In the last notebook we met logistic regression, $k$NN, decision boundaries, the confusion matrix, and precision/recall on toy data. Now we use them on a classic problem in galaxy evolution: separating quiescent from star-forming galaxies.

The astronomy. Galaxies at $0.5 < z < 2.5$ fall into two populations: galaxies still forming stars, and quiescent galaxies whose star formation has shut down. The standard way to separate them uses two rest-frame colours, $U-V$ and $V-J$ (the *UVJ diagram*). Quiescent galaxies are red in $U-V$ but relatively blue in $V-J$; dusty star-forming galaxies are red in both. This breaks the age–dust degeneracy that a single colour cannot. The usual selection is a hand-drawn "wedge" (Williams et al. 2009).

The ML version. Instead of drawing the boundary by hand, we learn it:
* labels — quiescent vs. star-forming from the specific star formation rate, $\mathrm{sSFR} = \mathrm{SFR}/M_*$, with SFRs from UV + Spitzer/MIPS 24 µm (Whitaker et al. 2014) and masses from FAST,
* features — the rest-frame $U-V$ and $V-J$ colours from EAZY (Brammer et al. 2011),

and compare the learned boundary with the classic wedge.

We use the 3D-HST catalogue for the COSMOS field (Skelton et al. 2014).

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q astropy gdown

import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import gdown
import matplotlib.pyplot as plt
from astropy.table import Table

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve)

SF_COLOR, Q_COLOR = "C0", "C1"              # star-forming, quiescent

plt.rcParams.update({
    "figure.figsize": (7, 5), "figure.dpi": 110,
    "font.size": 12, "axes.titlesize": 13,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.8,
    "lines.linewidth": 2, "legend.frameon": False,
})

## 1. Load the catalogue

We load the pre-joined 3D-HST COSMOS table (`cosmos_3dhst_v4.1_joined.fits`) from Google Drive. (The full pipeline builds this from two archives at MAST and joins them on `id` — we skip that here to save time.)

In [ ]:
DRIVE_FILE_ID = "17QVadZmdgfuerZMYmJWc4Bn22swCps03" 

DATA = Path(".")
fits_path = DATA / "cosmos_3dhst_v4.1_joined.fits"
if not fits_path.exists():
    gdown.download(id=DRIVE_FILE_ID, output=str(fits_path), quiet=False)

df = Table.read(fits_path).to_pandas()
print(f"{len(df)} sources loaded")
df.describe().T[["count", "mean", "min", "max"]].round(2)

What are the columns?

* `use_phot = 1` — the standard clean-photometry flag: good F125W/F160W coverage, not near a bright star, not a star (see Skelton et al. 2014).
* `z` — the redshift used for the rest-frame colours: the spectroscopic redshift `z_spec` if available, else the EAZY photo-z `z_peak`. `z_spec = -1` means no spectroscopic redshift.
* `lmass` — $\log_{10} M_*/M_\odot$ from FAST.
* `l153`, `l155`, `l161` — rest-frame flux densities in the Johnson $U$, $V$ and 2MASS $J$ filters (AB zero point 25).
* `sfr` — total UV+IR star formation rate in $M_\odot\,\mathrm{yr}^{-1}$, from Whitaker et al.'s 24 μm + UV recipe.
* `f24tot`, `ef24tot` — the 24 μm aperture flux density and its uncertainty; `f24tot / ef24tot` is the 24 μm detection significance (S/N) that the IR part of `sfr` is built from.

## 2. The sample and the labels

We keep galaxies with clean photometry at $0.5 \le z \le 2.5$ and $\log M_*/M_\odot \ge 10$. The mass cut keeps the sample complete in both classes: at low masses the survey would miss faint, red quiescent galaxies before it misses blue star-forming ones, and our classifier would learn that selection effect instead of physics. The upper redshift limit keeps us in the range where the 24 µm SFRs are most reliable.

Caveat — 24 μm non-detections treated as measurements. Faint quiescent galaxies are rarely detected by MIPS at 24 μm. The sample cut `sfr > 0` already removes the clearest non-detections (negative 24 μm flux, where the catalogue sets `sfr = -99`), but galaxies detected at low significance still get a finite "IR" SFR built from a noise-dominated 24 μm flux. That SFR is not reliable, and using it as-is can inflate the total SFR of a galaxy that is actually quiescent, landing it in the star-forming class. This is worst at $z > 1$, where 24 μm probes rest-frame mid-IR features and detections are shallower.

The fix used below: define the 24 μm significance $\mathrm{S/N} = f_{24,\mathrm{tot}} / \sigma_{f_{24,\mathrm{tot}}}$, and only trust the UV+IR `sfr` when $\mathrm{S/N} \ge 3$; otherwise fall back to the UV-only SFR (`sfr_uv`). (A more rigorous alternative is to propagate the 24 μm flux limit itself to an upper limit on the IR — and hence total — SFR, and call the galaxy quiescent if even that upper limit falls below the sSFR cut. We use the simpler UV-only fallback here.)

In [ ]:
Z_MIN, Z_MAX, MASS_MIN = 0.5, 2.5, 9.0

sel = ((df.use_phot == 1) & (df.z >= Z_MIN) & (df.z <= Z_MAX) & (df.lmass >= MASS_MIN)
       & (df.sfr > 0) & (df.l153 > 0) & (df.l155 > 0) & (df.l161 > 0))
d = df[sel].copy()
print(f"{len(d)} galaxies with use_phot=1, {Z_MIN} ≤ z ≤ {Z_MAX}, log M* ≥ {MASS_MIN}")
print(f"  of which {np.sum(d.z_spec > 0)} have a spectroscopic redshift")

### ✏️ Exercise 1 — Features and labels

Add four columns to `d`:

1. `UV` — the rest-frame colour $U - V = -2.5 \log_{10}(L_{153}/L_{155})$
2. `VJ` — the rest-frame colour $V - J = -2.5 \log_{10}(L_{155}/L_{161})$
3. `log_ssfr` — $\log_{10}(\mathrm{SFR}/M_*)$ in $\mathrm{yr}^{-1}$ (remember that `lmass` is already a logarithm). 
4. `quiescent` — 1 if `log_ssfr < LOG_SSFR_CUT`, else 0

In [ ]:
LOG_SSFR_CUT = -10.0      # yr^-1: roughly an order of magnitude below the star-forming main sequence at these z

# YOUR CODE HERE
# d["UV"] = 
# d["VJ"] = 
# d["log_ssfr"] = 
# d["quiescent"] = 


# Only trust the IR part of "sfr" when the 24um detection is significant; otherwise the IR term
# is noise-dominated and can inflate the SFR of a genuinely quiescent galaxy. Fall back to the
# UV-only SFR for low-significance 24um detections.
d["sn24"] = d.f24tot / d.ef24tot
d["sfr_label"] = np.where(d.sn24 >= 3, d.sfr, d.sfr_uv)    # UV+IR if detected at 24um, else UV only

print(f"quiescent fraction: {d.quiescent.mean():.1%}  ({d.quiescent.sum()} of {len(d)})")
low_sn24 = d.sn24 < 3
print(f"  {low_sn24.sum()} galaxies ({low_sn24.mean():.1%}) have a low-significance 24μm "
      f"detection (S/N < 3) and fall back to the UV-only SFR")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
ax1.hexbin(d.lmass, d.log_ssfr, gridsize=45, bins="log", mincnt=1, cmap="Blues",
           extent=(10, 11.8, -12.5, -7.5))
ax1.axhline(LOG_SSFR_CUT, color=Q_COLOR, ls="--", lw=1.5, label=f"cut: log sSFR = {LOG_SSFR_CUT}")
ax1.set(xlabel=r"$\log\,M_*/M_\odot$", ylabel=r"$\log$ sSFR [yr$^{-1}$]", ylim=(-12.5, -7.5),
        title="UV+IR specific SFR vs. stellar mass")
ax1.legend(loc="lower left")

bins = np.arange(-12.5, -7.49, 0.1)
for (lo, hi), col in zip([(0.5, 1.0), (1.0, 1.5), (1.5, 2.5)], ["C0", "C1", "C2"]):
    s = (d.z >= lo) & (d.z < hi)
    ax2.hist(d.log_ssfr[s], bins=bins, histtype="step", lw=2, color=col, density=True,
             label=f"{lo} ≤ z < {hi}")
ax2.axvline(LOG_SSFR_CUT, color=Q_COLOR, ls="--", lw=1.5)
ax2.set(xlabel=r"$\log$ sSFR [yr$^{-1}$]", ylabel="normalised count", title="Is the distribution bimodal?")
ax2.legend(loc="upper left");

In [ ]:
# The classic UVJ wedge (Williams et al. 2009). Several slightly different versions are in use;
# change the numbers here to try another.
WEDGE = dict(uv_min=1.3, vj_max=1.6, slope=0.88, intercept=0.59)

def draw_wedge(ax, w=WEDGE, **kw):
    vj0 = (w["uv_min"] - w["intercept"]) / w["slope"]          # where the diagonal meets U-V = uv_min
    uv1 = w["slope"]*w["vj_max"] + w["intercept"]               # where the diagonal meets V-J = vj_max
    style = dict(color="black", lw=2) | kw
    ax.plot([-1, vj0, w["vj_max"], w["vj_max"]], [w["uv_min"], w["uv_min"], uv1, 3], **style)

def uvj_axes(ax, title=""):
    ax.set(xlim=(-0.3, 2.3), ylim=(0.2, 2.6), xlabel="V − J  (rest-frame)",
           ylabel="U − V  (rest-frame)", title=title)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)
for ax, cls, col, name in zip(axes, [0, 1], [SF_COLOR, Q_COLOR], ["star-forming", "quiescent"]):
    ax.scatter(d.VJ, d.UV, s=2, color="gray", alpha=0.15, rasterized=True)
    s = d.quiescent == cls
    ax.scatter(d.VJ[s], d.UV[s], s=4, color=col, alpha=0.6, rasterized=True)
    draw_wedge(ax)
    uvj_axes(ax, f"{name} by sSFR  (N = {s.sum()})")
axes[1].set_ylabel("")
fig.tight_layout();

The sSFR labels and the UVJ colours are independent measurements: one comes from UV + mid-IR luminosities, the other from rest-frame optical/near-IR template fits. Yet the quiescent galaxies pile up in the wedge. That is why the UVJ diagram works.

In [ ]:
X = d[["VJ", "UV"]].to_numpy()
y = d["quiescent"].to_numpy()
idx_train, idx_test = train_test_split(np.arange(len(d)), test_size=0.3, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = X[idx_train], X[idx_test], y[idx_train], y[idx_test]
print(f"train: {len(y_train)}   test: {len(y_test)}   quiescent fraction: {y.mean():.1%}")

## 3. The UVJ wedge as a classifier

The wedge is itself a classifier — just one whose boundary was drawn by hand. A galaxy is classified as quiescent if

$$U-V > 1.3, \qquad V-J < 1.6, \qquad U-V > 0.88\,(V-J) + 0.59 .$$

Let's measure it with the tools from the last session. We treat quiescent as the positive class, so

* precision (purity) — what fraction of the galaxies in the wedge are really quiescent (by sSFR)?
* recall (completeness) — what fraction of the quiescent galaxies land in the wedge?

### ✏️ Exercise 2 — Score the wedge

1. Complete `uvj_quiescent(uv, vj)`, which returns a boolean array: `True` inside the wedge. Use the numbers in the `WEDGE` dictionary.
2. Apply it to the test set (column 0 of `X_test` is $V-J$, column 1 is $U-V$) to get `y_wedge`.
3. The cell then prints accuracy, precision, recall and $F_1$, and shows the confusion matrix.

In [ ]:
def uvj_quiescent(uv, vj, w=WEDGE):
    return (uv > w["uv_min"]) & (vj < w["vj_max"]) & (uv > w["slope"]*vj + w["intercept"])

# YOUR CODE HERE
# y_wedge = 


def scores(y_true, y_pred):
    return dict(accuracy=accuracy_score(y_true, y_pred), precision=precision_score(y_true, y_pred),
                recall=recall_score(y_true, y_pred), F1=f1_score(y_true, y_pred))

results = {"UVJ wedge": scores(y_test, y_wedge)}
print({k: round(v, 3) for k, v in results["UVJ wedge"].items()})

fig, ax = plt.subplots(figsize=(4.8, 4.3))
ConfusionMatrixDisplay.from_predictions(y_test, y_wedge, display_labels=["star-forming", "quiescent"],
                                        cmap="Blues", colorbar=False, ax=ax)
ax.grid(False)
ax.set(title="UVJ wedge vs. sSFR labels", xlabel="predicted (wedge)", ylabel="true (sSFR)");

## 4. Learning the boundary

Now let the data decide where the boundary goes. Rather than just writing the fit-and-score code once, here we fix the code and instead change the model's parameters by hand, one model at a time — logistic regression, $k$NN, SVM, and a random forest — looking at how the boundary and the scores respond. At the end we bring all four together.

Bonus (if you have time): scikit-learn can automate this kind of parameter search instead of trying values by hand — `validation_curve` sweeps one parameter and returns train/test scores for each value (exactly what you're about to do manually below), while `GridSearchCV` / `RandomizedSearchCV` search over several parameters at once using cross-validation and report the best combination. Try replacing one of the manual exercises below with one of these.

In [ ]:
# Shared setup for Section 4: a grid over the UVJ plane, and a helper to plot
# one model's P(quiescent) map against the test points and the wedge.
gv, gu = np.meshgrid(np.linspace(-0.3, 2.3, 300), np.linspace(0.2, 2.6, 300))
grid = np.c_[gv.ravel(), gu.ravel()]

def plot_boundary(model, title):
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    p = model.predict_proba(grid)[:, 1].reshape(gv.shape)
    ax.contourf(gv, gu, p, levels=np.linspace(0, 1, 11), cmap="Oranges", alpha=0.35)
    ax.scatter(X_test[y_test == 0, 0], X_test[y_test == 0, 1], s=3, color=SF_COLOR, alpha=0.4,
               rasterized=True, label="star-forming (sSFR)")
    ax.scatter(X_test[y_test == 1, 0], X_test[y_test == 1, 1], s=5, color=Q_COLOR, alpha=0.7,
               rasterized=True, label="quiescent (sSFR)")
    ax.contour(gv, gu, p, levels=[0.5], colors="black", linewidths=2, linestyles="--")
    draw_wedge(ax, lw=1.5, color="gray")
    uvj_axes(ax, f"{title}: dashed = P(quiescent) = 0.5, grey = wedge")
    ax.legend(loc="upper left", markerscale=4)
    ax.grid(False)
    plt.show()

### ✏️ Exercise 3a — Logistic regression: `C` and `class_weight`

`LogisticRegression`'s `C` is the inverse regularization strength: small `C` → strong regularization → the model shrinks its coefficients and draws a simpler boundary; large `C` → weak regularization → it fits the training points more closely, at the risk of overfitting.

`class_weight` controls how much each class counts towards the loss. Our sample is imbalanced (~29% quiescent). With `class_weight=None`, misclassifying the majority (star-forming) and minority (quiescent) classes count equally; with `class_weight="balanced"`, the minority class is upweighted so both classes effectively matter equally — this trades some accuracy for recall on quiescent galaxies.

Try `C_VALUE` in 0.01, 1, 100 and `CLASS_WEIGHT` in None, "balanced", in any combination, and re-run. Which one moves the boundary more? Which one moves precision/recall more?

In [ ]:
C_VALUE = 1     # try 0.01, 1, 100
CLASS_WEIGHT = None  # try None, "balanced"

logreg = make_pipeline(StandardScaler(), LogisticRegression(C=C_VALUE, class_weight=CLASS_WEIGHT)).fit(X_train, y_train)
y_logreg = logreg.predict(X_test)
results["logistic regression"] = scores(y_test, y_logreg)

print({k: round(v, 3) for k, v in results["logistic regression"].items()})
plot_boundary(logreg, f"Logistic regression, C = {C_VALUE}, class_weight = {CLASS_WEIGHT}")

### ✏️ Exercise 3b — $k$NN: `n_neighbors` and `weights`

Same idea as the regression notebook: `n_neighbors` controls flexibility. Small `k` → a jagged boundary that follows individual points (can overfit); large `k` → a smooth boundary that averages over more of the data (can underfit).

`weights` controls how the $k$ neighbours are combined. With `"uniform"`, all $k$ neighbours count equally; with `"distance"`, closer neighbours count more — this lets a large `k` still respond sharply to the nearest points instead of over-smoothing.

Try `K_VALUE` in 5, 30, 100 and `WEIGHTS` in "uniform", "distance", and re-run. Does `"distance"` weighting let a large `k` behave more like a small one?

In [ ]:
K_VALUE = 100          # try 5, 30, 100
WEIGHTS = "uniform"   # try "uniform", "distance"

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=K_VALUE, weights=WEIGHTS)).fit(X_train, y_train)
results["kNN"] = scores(y_test, knn.predict(X_test))

print({k: round(v, 3) for k, v in results["kNN"].items()})
plot_boundary(knn, f"kNN, k = {K_VALUE}, weights = {WEIGHTS}")

### ✏️ Exercise 3c — SVM: `gamma` and `C`

An RBF-kernel SVM has two main knobs. `gamma` sets how far a single training point's influence reaches: small `gamma` → influence reaches far → a smoother, almost-linear boundary; large `gamma` → influence is very local → the boundary can wrap tightly around individual points (overfitting).

`C` plays a similar role here to logistic regression's `C`: small `C` allows more training points to be misclassified in exchange for a wider margin (simpler boundary); large `C` penalizes every training mistake heavily, pulling the boundary tight around the data (overfitting).

Try `GAMMA_VALUE` in 0.1, 1, 10, "scale" and `C_VALUE` in 0.1, 1, 10, and re-run. Can a small `gamma` and a large `C` cancel each other out?

In [ ]:
GAMMA_VALUE = "scale"   # try 0.1, 1, 10, or "scale"
C_VALUE = 1.0           # try 0.1, 1, 10

svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", gamma=GAMMA_VALUE, C=C_VALUE, probability=True)).fit(X_train, y_train)
results["SVM (RBF)"] = scores(y_test, svm.predict(X_test))

print({k: round(v, 3) for k, v in results["SVM (RBF)"].items()})
plot_boundary(svm, f"SVM (RBF), gamma = {GAMMA_VALUE}, C = {C_VALUE}")

### ✏️ Exercise 3d — Random forest: `max_depth` and `n_estimators`

A random forest averages many decision trees. `max_depth` limits how many splits each tree can make: shallow trees (small `max_depth`) underfit — a coarse, blocky boundary; deep trees (large `max_depth`, or `None` for unlimited) can fit the training set almost perfectly, which risks overfitting on a dataset this size.

`n_estimators` is the number of trees averaged together. Unlike `max_depth`, adding more trees does not make the forest more prone to overfitting — it mainly reduces the *variance* from the randomness in how each tree is built, so the boundary gets smoother and more stable (at the cost of more computation).

Try `MAX_DEPTH` in 2, 5, None and `N_ESTIMATORS` in 10, 300, and re-run. Does increasing `n_estimators` change the score the way increasing `max_depth` does?

In [ ]:
MAX_DEPTH = 5        # try 2, 5, None
N_ESTIMATORS = 300   # try 10, 300

rf = RandomForestClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, random_state=0).fit(X_train, y_train)
results["random forest"] = scores(y_test, rf.predict(X_test))

print({k: round(v, 3) for k, v in results["random forest"].items()})
plot_boundary(rf, f"Random forest, max_depth = {MAX_DEPTH}, n_estimators = {N_ESTIMATORS}")

### Bringing it together

All four learned boundaries, scored on the same test set:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5.3), sharey=True)
for ax, model, name in zip(axes, [logreg, knn, svm, rf],
                           ["Logistic regression", "kNN", "SVM (RBF)", "Random forest"]):
    p = model.predict_proba(grid)[:, 1].reshape(gv.shape)
    ax.contourf(gv, gu, p, levels=np.linspace(0, 1, 11), cmap="Oranges", alpha=0.35)
    ax.scatter(X_test[y_test == 0, 0], X_test[y_test == 0, 1], s=2, color=SF_COLOR, alpha=0.4, rasterized=True)
    ax.scatter(X_test[y_test == 1, 0], X_test[y_test == 1, 1], s=3, color=Q_COLOR, alpha=0.7, rasterized=True)
    ax.contour(gv, gu, p, levels=[0.5], colors="black", linewidths=2, linestyles="--")
    draw_wedge(ax, lw=1.5, color="gray")
    uvj_axes(ax, name)
    ax.grid(False)
for ax in axes[1:]:
    ax.set_ylabel("")
fig.tight_layout()

pd.DataFrame(results).T.round(3)

Bonus (if you have time): the wedge only uses two colours. Add redshift and stellar mass as extra features (use `d[["VJ", "UV", "z", "lmass"]]` with the same `idx_train` / `idx_test`) and refit the logistic regression. Does it help? Why might the boundary depend on redshift?

## 5. Purity vs. completeness

A hand-drawn wedge gives you one trade-off between purity and completeness. A probabilistic classifier gives you the whole curve, and you choose the operating point for your science case. A study of the *most* passive galaxies needs a pure sample; a measurement of the quiescent fraction or the stellar mass function needs a complete one.

In [ ]:
p_test = logreg.predict_proba(X_test)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, p_test)

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.plot(rec, prec, color="C0", label="logistic regression, all thresholds")
w = results["UVJ wedge"]
ax.plot(w["recall"], w["precision"], "s", ms=10, color="black", label="UVJ wedge")
lr = results["logistic regression"]
ax.plot(lr["recall"], lr["precision"], "o", ms=10, color=Q_COLOR, label="logistic regression, P > 0.5")
ax.axhline(y_test.mean(), color="gray", ls="--", lw=1, label="random guessing")
ax.set(xlabel="completeness (recall)", ylabel="purity (precision)", xlim=(0, 1.02), ylim=(0, 1.05),
       title="Selecting quiescent galaxies")
ax.legend(loc="lower left");

## 6. Does redshift quality matter?

The rest-frame colours (and the stellar masses, and the SFRs) all depend on the redshift. In the first lecture we saw that photometric redshifts have scatter and catastrophic outliers. If the redshift is wrong, the rest-frame $U$, $V$ and $J$ fluxes are interpolated from the wrong part of the SED.

Part of our sample has a spectroscopic redshift (`z_spec > 0`; the cell in Section 2 printed how many). For those, the rest-frame colours were computed at the spectroscopic redshift; for the rest, at the photo-z.

### ✏️ Exercise 4 — Spectroscopic vs. photometric redshifts

1. Make a boolean mask `has_zspec` for the test set (use `d.z_spec.to_numpy()[idx_test]`).
2. Compute the scores of the wedge and of the logistic regression separately for test galaxies with and without a spectroscopic redshift, and store them in the dictionary `by_z`, e.g. `by_z["logreg, spec-z"] = scores(...)`.

Is the classification better for galaxies with spectroscopic redshifts? Before you conclude that photo-z errors are to blame, compare the two subsamples: how else might galaxies with spectra differ from those without?

In [ ]:
by_z = {}

has_zspec = d.z_spec.to_numpy()[idx_test] > 0
for name, pred in [("wedge", y_wedge), ("logreg", y_logreg)]:
    by_z[f"{name}, spec-z"] = scores(y_test[has_zspec], pred[has_zspec])
    by_z[f"{name}, photo-z"] = scores(y_test[~has_zspec], pred[~has_zspec])

print(f"test galaxies with spec-z: {has_zspec.sum()},  photo-z only: {(~has_zspec).sum()}")
pd.DataFrame(by_z).T.round(3)

In [ ]:
# How do the two subsamples differ?
t = d.iloc[idx_test]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, bins, lab in zip(axes, ["z", "lmass", "quiescent"],
                              [np.arange(0.5, 2.51, 0.1), np.arange(10, 11.81, 0.1), [-0.5, 0.5, 1.5]],
                              ["redshift", r"$\log M_*/M_\odot$", "quiescent (sSFR label)"]):
    ax.hist(t[col][has_zspec], bins=bins, density=True, histtype="step", lw=2, color="C0", label="spec-z")
    ax.hist(t[col][~has_zspec], bins=bins, density=True, histtype="step", lw=2, color="C1", label="photo-z only")
    ax.set(xlabel=lab, ylabel="normalised count")
axes[2].set_xticks([0, 1], ["star-forming", "quiescent"])
axes[0].legend()
fig.tight_layout();

## 7. Discussion and summary

* The UVJ wedge is a hand-made linear classifier, and it holds up well against independent sSFR labels. A learned boundary is similar, and gives calibrated probabilities that let you choose purity vs. completeness.
* The limiting factor is not the flexibility of the model: it is the labels (UV+IR SFRs of quiescent galaxies are often upper limits or contaminated by AGN and old stars) and the features (rest-frame colours inherit photo-z and template errors).
* Sample selection matters twice: the mass cut keeps the classes complete, and spectroscopic subsamples are biased in ways that change the metrics.